## 1. Setup

The main dependencies are pandas, NumPy, scikit-learn, and PyTorch. 

In [1]:
%pip install -q transformers

In [2]:
from pathlib import Path
import json, random, re, time, os
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             accuracy_score, hamming_loss)
from IPython.display import display

SEED = 42
CSV_PATH = Path("complete-dataset.csv")
OUT_DIR = Path("experiment_results")
OUT_DIR.mkdir(exist_ok=True)
HF_HOME = Path(".venv/huggingface").resolve()
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME / "hub")
LABELS = ["Computer Science", "Physics", "Mathematics", "Statistics",
          "Quantitative Biology", "Quantitative Finance"]
MAX_LEN, VOCAB_LIMIT, EMBED_DIM, HIDDEN = 192, 50_000, 300, 80
BATCH_SIZE, EPOCHS = 96, 5
TOKEN_RE = re.compile(r"(?u)[\w]+(?:[-'][\w]+)*")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
                      else "cpu")

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(min(4, torch.get_num_threads()))

seed_everything()
print("Device:", DEVICE)
print("CSV found:", CSV_PATH.exists())

Device: cuda
CSV found: True


## 2. Load and inspect the dataset

In [3]:
if not CSV_PATH.exists():
    try:
        from google.colab import files
        print("Upload complete-dataset.csv")
        files.upload()
    except ImportError as exc:
        raise FileNotFoundError(f"Place {CSV_PATH.name} beside this notebook") from exc
    if not CSV_PATH.exists():
        raise FileNotFoundError(f"{CSV_PATH.name} was not uploaded")

df = pd.read_csv(CSV_PATH)
required = ["TITLE", "ABSTRACT", *LABELS]
missing_columns = [col for col in required if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# The supplied file has no missing values; fill defensively for repeatability.
df["TITLE"] = df["TITLE"].fillna("").astype(str)
df["ABSTRACT"] = df["ABSTRACT"].fillna("").astype(str)
df = df.loc[(df["TITLE"] + df["ABSTRACT"]).str.strip().ne("")].reset_index(drop=True)
texts = (df["TITLE"] + " [SEP] " + df["ABSTRACT"]).tolist()
y = df[LABELS].astype(np.uint8).to_numpy()

print("Rows:", len(df), "| duplicate title/abstract pairs:", df.duplicated(["TITLE", "ABSTRACT"]).sum())
display(pd.DataFrame({"positive_count": y.sum(axis=0), "positive_rate": y.mean(axis=0)}, index=LABELS))
print("Mean labels per paper:", y.sum(axis=1).mean().round(3))

Rows: 20972 | duplicate title/abstract pairs: 0


,positive_count,positive_rate
Computer Science,8594,0.409784
Physics,6013,0.286716
Mathematics,5618,0.267881
Statistics,5206,0.248236
Quantitative Biology,587,0.027990
Quantitative Finance,249,0.011873


Mean labels per paper: 1.252


## 3. Fixed multilabel train/validation/test split

A deterministic greedy stratification assigns 80% to training, 10% to validation, and 10% to test. The test set is reserved for final reporting.

In [4]:
def iterative_multilabel_split(y, fractions=(.8, .1, .1), seed=SEED):
    """Greedy multilabel split that prioritizes rare positive labels."""
    rng = np.random.default_rng(seed)
    n, _ = y.shape
    target_n = np.floor(np.asarray(fractions) * n).astype(int)
    target_n[-1] = n - target_n[:-1].sum()
    target_pos = np.outer(fractions, y.sum(axis=0))
    assigned_pos = np.zeros_like(target_pos)
    remaining = target_n.copy()
    label_counts = np.maximum(y.sum(axis=0), 1)
    jitter = rng.random(n)
    rarity = np.where(y > 0, 1 / label_counts, 0).sum(axis=1)
    order = np.lexsort((jitter, -rarity))
    bins = [[] for _ in fractions]
    for idx in order:
        active = np.flatnonzero(y[idx])
        candidates = np.flatnonzero(remaining > 0)
        if len(active):
            demand = target_pos[candidates][:, active] - assigned_pos[candidates][:, active]
            score = (demand / np.maximum(target_pos[candidates][:, active], 1)).sum(axis=1)
        else:
            score = np.zeros(len(candidates))
        score += remaining[candidates] / max(n, 1) * 1e-3
        chosen = candidates[int(np.argmax(score))]
        bins[chosen].append(int(idx))
        remaining[chosen] -= 1
        assigned_pos[chosen] += y[idx]
    return [np.asarray(items, dtype=np.int64) for items in bins]

train_idx, val_idx, test_idx = iterative_multilabel_split(y)
print("Split sizes:", len(train_idx), len(val_idx), len(test_idx))
display(pd.DataFrame({"train": y[train_idx].sum(axis=0),
                      "validation": y[val_idx].sum(axis=0),
                      "test": y[test_idx].sum(axis=0)}, index=LABELS))

Split sizes: 16777 2097 2098


,train,validation,test
Computer Science,6876,858,860
Physics,4809,602,602
Mathematics,4494,562,562
Statistics,4164,521,521
Quantitative Biology,469,59,59
Quantitative Finance,199,25,25


## 4. Metrics and validation-only threshold selection

In [5]:
def tune_thresholds(y_true, probabilities):
    grid = np.arange(.10, .91, .02)
    thresholds = []
    for j in range(y_true.shape[1]):
        scores = [f1_score(y_true[:, j], probabilities[:, j] >= t, zero_division=0) for t in grid]
        thresholds.append(float(grid[int(np.argmax(scores))]))
    return np.asarray(thresholds)

def evaluate(y_true, probabilities, thresholds):
    pred = (probabilities >= np.asarray(thresholds)[None, :]).astype(np.uint8)
    overall = {
        "micro_precision": precision_score(y_true, pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, pred, average="micro", zero_division=0),
        "micro_f1": f1_score(y_true, pred, average="micro", zero_division=0),
        "macro_precision": precision_score(y_true, pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "exact_match_accuracy": accuracy_score(y_true, pred),
        "hamming_loss": hamming_loss(y_true, pred),
    }
    per_class = {label: {
        "precision": precision_score(y_true[:, j], pred[:, j], zero_division=0),
        "recall": recall_score(y_true[:, j], pred[:, j], zero_division=0),
        "f1": f1_score(y_true[:, j], pred[:, j], zero_division=0),
        "support": int(y_true[:, j].sum()),
        "threshold": float(thresholds[j]),
    } for j, label in enumerate(LABELS)}
    return {**overall, "per_class": per_class}

def save_prob_result(name, val_probs, test_probs):
    thresholds = tune_thresholds(y[val_idx], val_probs)
    result = {
        "fixed_0_5": evaluate(y[test_idx], test_probs, np.full(len(LABELS), .5)),
        "validation_tuned": evaluate(y[test_idx], test_probs, thresholds),
        "thresholds": dict(zip(LABELS, thresholds.tolist())),
    }
    results[name] = result
    return result

results = {}

## 5. Baseline: TF–IDF + one-vs-rest logistic regression

In [6]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=150_000,
                             sublinear_tf=True, strip_accents="unicode", dtype=np.float32)
x_train = vectorizer.fit_transform([texts[i] for i in train_idx])
x_val = vectorizer.transform([texts[i] for i in val_idx])
x_test = vectorizer.transform([texts[i] for i in test_idx])
baseline = OneVsRestClassifier(LogisticRegression(C=4, max_iter=300, solver="liblinear"), n_jobs=1)
baseline.fit(x_train, y[train_idx])
baseline_result = save_prob_result("TF-IDF + logistic regression",
                                   baseline.predict_proba(x_val), baseline.predict_proba(x_test))
print("Best baseline test macro F1:", round(baseline_result["validation_tuned"]["macro_f1"], 4))

Best baseline test macro F1: 0.7502


## 6. Shared recurrent preprocessing and model

In [7]:
def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())

tokenized = [tokenize(text) for text in texts]
counts = Counter(token for i in train_idx for token in tokenized[i])
vocab = {"<PAD>": 0, "<UNK>": 1}
vocab.update({token: i + 2 for i, (token, _) in enumerate(counts.most_common(VOCAB_LIMIT - 2))})
sequences = np.zeros((len(texts), MAX_LEN), dtype=np.int32)
lengths = np.zeros(len(texts), dtype=np.int32)
for i, tokens in enumerate(tokenized):
    ids = [vocab.get(token, 1) for token in tokens[:MAX_LEN]] or [1]
    sequences[i, :len(ids)] = ids
    lengths[i] = len(ids)
print("Training-only vocabulary size:", len(vocab))

# Download the official 300-D English FastText subword vectors if they are not cached.
# In Colab, the archive is stored under /content; locally it is stored under .venv/models.
import io, zipfile
FASTTEXT_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M-subword.vec.zip"
LOCAL_FASTTEXT = Path(".venv/models/fasttext/wiki-news-300d-1M-subword.vec.zip")
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
FASTTEXT_ARCHIVE = (Path("/content/wiki-news-300d-1M-subword.vec.zip")
                    if IN_COLAB else LOCAL_FASTTEXT)
if not FASTTEXT_ARCHIVE.exists():
    import requests
    from tqdm.auto import tqdm
    FASTTEXT_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    partial = FASTTEXT_ARCHIVE.with_suffix(FASTTEXT_ARCHIVE.suffix + ".part")
    response = requests.get(FASTTEXT_URL, stream=True, timeout=60)
    response.raise_for_status()
    total = int(response.headers.get("content-length", 0))
    with partial.open("wb") as output, tqdm(total=total, unit="B", unit_scale=True,
                                             desc="FastText vectors") as progress:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                output.write(chunk)
                progress.update(len(chunk))
    partial.replace(FASTTEXT_ARCHIVE)
print("FastText archive:", FASTTEXT_ARCHIVE)

def load_fasttext_matrix(archive_path, word_to_id, dimension=EMBED_DIM):
    matrix = np.random.normal(0, .05, (len(word_to_id), dimension)).astype(np.float32)
    matrix[word_to_id["<PAD>"]] = 0
    found = set()
    with zipfile.ZipFile(archive_path) as archive:
        members = [name for name in archive.namelist() if name.endswith(".vec")]
        if not members:
            raise ValueError("The FastText archive contains no .vec file")
        with archive.open(members[0]) as binary:
            text = io.TextIOWrapper(binary, encoding="utf-8", errors="ignore")
            header = text.readline().split()
            if len(header) != 2 or int(header[1]) != dimension:
                raise ValueError(f"Expected {dimension}-D FastText vectors; header was {header}")
            for line in text:
                parts = line.rstrip().split(" ")
                word = parts[0]
                row = word_to_id.get(word)
                if row is None or len(parts) != dimension + 1:
                    continue
                matrix[row] = np.asarray(parts[1:], dtype=np.float32)
                found.add(row)
    matrix[word_to_id["<PAD>"]] = 0
    return matrix, found

embedding_matrix, fasttext_found = load_fasttext_matrix(FASTTEXT_ARCHIVE, vocab)
fasttext_matches = len(fasttext_found - {vocab["<UNK>"]})
fasttext_total = len(vocab) - 2
print(f"FastText coverage: {fasttext_matches:,}/{fasttext_total:,} tokens "
      f"({fasttext_matches/max(fasttext_total, 1):.1%}); remaining rows initialized randomly")

class PaperDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices)
    def __len__(self): return len(self.indices)
    def __getitem__(self, k):
        i = self.indices[k]
        return (torch.tensor(sequences[i], dtype=torch.long),
                torch.tensor(lengths[i], dtype=torch.long),
                torch.tensor(y[i], dtype=torch.float32))

class RecurrentClassifier(nn.Module):
    def __init__(self, kind):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.from_numpy(embedding_matrix), freeze=False, padding_idx=0)
        layer = nn.LSTM if kind == "BiLSTM" else nn.GRU
        self.rnn = layer(EMBED_DIM, HIDDEN, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(.35)
        self.output = nn.Linear(2 * HIDDEN, len(LABELS))
    def forward(self, tokens, token_lengths):
        embedded = self.embedding(tokens)
        packed = pack_padded_sequence(embedded, token_lengths.cpu(), batch_first=True,
                                      enforce_sorted=False)
        _, state = self.rnn(packed)
        hidden = state[0] if isinstance(state, tuple) else state
        pooled = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.output(self.dropout(pooled))

Training-only vocabulary size: 50000


FastText vectors:   0%|          | 0.00/588M [00:00<?, ?B/s]

FastText archive: /content/wiki-news-300d-1M-subword.vec.zip
FastText coverage: 30,342/49,998 tokens (60.7%); remaining rows initialized randomly


## 7. Train BiLSTM and BiGRU

In [8]:
def train_recurrent(kind, epochs=EPOCHS):
    train_loader = DataLoader(PaperDataset(train_idx), batch_size=BATCH_SIZE, shuffle=True,
                              generator=torch.Generator().manual_seed(SEED))
    val_loader = DataLoader(PaperDataset(val_idx), batch_size=BATCH_SIZE)
    test_loader = DataLoader(PaperDataset(test_idx), batch_size=BATCH_SIZE)
    model = RecurrentClassifier(kind).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    best_score, best_state, history = -1, None, []
    for epoch in range(1, epochs + 1):
        start = time.time(); model.train(); losses=[]
        for tokens, token_lengths, labels in train_loader:
            tokens, labels = tokens.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(tokens, token_lengths), labels)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); losses.append(float(loss.detach().cpu()))
        model.eval(); val_probs=[]
        with torch.no_grad():
            for tokens, token_lengths, _ in val_loader:
                val_probs.append(torch.sigmoid(model(tokens.to(DEVICE), token_lengths)).cpu().numpy())
        val_probs = np.concatenate(val_probs)
        score = f1_score(y[val_idx], val_probs >= .5, average="macro", zero_division=0)
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses)),
                        "val_macro_f1_at_0_5": float(score), "seconds": time.time()-start})
        print(f"{kind} epoch {epoch}/{epochs}: loss={np.mean(losses):.4f}; val macro F1={score:.4f}")
        if score > best_score:
            best_score = score
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    model.load_state_dict(best_state); model.eval()
    def predict(loader):
        chunks=[]
        with torch.no_grad():
            for tokens, token_lengths, _ in loader:
                chunks.append(torch.sigmoid(model(tokens.to(DEVICE), token_lengths)).cpu().numpy())
        return np.concatenate(chunks)
    val_probs, test_probs = predict(val_loader), predict(test_loader)
    torch.save({"state_dict": model.state_dict(), "vocab": vocab,
                "labels": LABELS, "max_length": MAX_LEN}, OUT_DIR / f"{kind.lower()}_checkpoint.pt")
    result = save_prob_result(kind, val_probs, test_probs)
    result["history"] = history
    return result

# Each run trains for five epochs on CPU in this environment; adjust EPOCHS if needed.
for architecture in ("BiLSTM", "BiGRU"):
    if architecture not in results:
        results[architecture] = train_recurrent(architecture)

BiLSTM epoch 1/5: loss=0.3961; val macro F1=0.4629
BiLSTM epoch 2/5: loss=0.2193; val macro F1=0.5269
BiLSTM epoch 3/5: loss=0.1564; val macro F1=0.5300
BiLSTM epoch 4/5: loss=0.1168; val macro F1=0.5230
BiLSTM epoch 5/5: loss=0.0847; val macro F1=0.5171
BiGRU epoch 1/5: loss=0.3821; val macro F1=0.4858
BiGRU epoch 2/5: loss=0.1961; val macro F1=0.4764
BiGRU epoch 3/5: loss=0.1197; val macro F1=0.4798
BiGRU epoch 4/5: loss=0.0734; val macro F1=0.4720
BiGRU epoch 5/5: loss=0.0441; val macro F1=0.4846


## 8. Fine-tune BERT

This cell uses `bert-base-uncased` with a six-logit multi-label classification head. It loads the project-local checkpoint and adds a new six-label classification head for fine-tuning.

In [9]:
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("BERT not run: transformers is missing; install it and provide the pretrained checkpoint.")

if TRANSFORMERS_AVAILABLE:
    LOCAL_BERT = Path(".venv/models/bert-base-uncased")
    BERT_NAME = str(LOCAL_BERT) if (LOCAL_BERT / "config.json").exists() else "bert-base-uncased"
    BERT_MAX_LEN, BERT_BATCH_SIZE, BERT_EPOCHS = 192, 8, 2
    tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)

    class BertPaperDataset(Dataset):
        def __init__(self, indices): self.indices = np.asarray(indices)
        def __len__(self): return len(self.indices)
        def __getitem__(self, k):
            i = self.indices[k]
            encoded = tokenizer(texts[i], max_length=BERT_MAX_LEN, truncation=True,
                                padding="max_length", return_tensors="pt")
            return ({key: val.squeeze(0) for key, val in encoded.items()},
                    torch.tensor(y[i], dtype=torch.float32))

    def train_bert():
        loaders = {
            "train": DataLoader(BertPaperDataset(train_idx), batch_size=BERT_BATCH_SIZE, shuffle=True),
            "validation": DataLoader(BertPaperDataset(val_idx), batch_size=BERT_BATCH_SIZE),
            "test": DataLoader(BertPaperDataset(test_idx), batch_size=BERT_BATCH_SIZE),
        }
        model = AutoModelForSequenceClassification.from_pretrained(
            BERT_NAME, num_labels=len(LABELS), problem_type="multi_label_classification").to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=.01)
        criterion = nn.BCEWithLogitsLoss()
        best_score, best_state = -1, None
        for epoch in range(1, BERT_EPOCHS + 1):
            model.train(); losses=[]
            for batch, labels in loaders["train"]:
                batch = {key: val.to(DEVICE) for key, val in batch.items()}
                labels = labels.to(DEVICE); optimizer.zero_grad(set_to_none=True)
                logits = model(**batch).logits; loss = criterion(logits, labels)
                loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
                losses.append(float(loss.detach().cpu()))
            model.eval(); val_probs=[]
            with torch.no_grad():
                for batch, _ in loaders["validation"]:
                    batch = {key: val.to(DEVICE) for key, val in batch.items()}
                    val_probs.append(torch.sigmoid(model(**batch).logits).cpu().numpy())
            val_probs = np.concatenate(val_probs)
            score = f1_score(y[val_idx], val_probs >= .5, average="macro", zero_division=0)
            print(f"BERT epoch {epoch}/{BERT_EPOCHS}: loss={np.mean(losses):.4f}; val macro F1={score:.4f}")
            if score > best_score:
                best_score = score
                best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        model.load_state_dict(best_state); model.eval()
        def predict(loader):
            chunks=[]
            with torch.no_grad():
                for batch, _ in loader:
                    batch = {key: val.to(DEVICE) for key, val in batch.items()}
                    chunks.append(torch.sigmoid(model(**batch).logits).cpu().numpy())
            return np.concatenate(chunks)
        val_probs, test_probs = predict(loaders["validation"]), predict(loaders["test"])
        model.save_pretrained(OUT_DIR / "bert_checkpoint")
        result = save_prob_result("BERT", val_probs, test_probs)
        return result

    if "BERT" not in results:
        results["BERT"] = train_bert()
else:
    results["BERT"] = {"status": "not_run", "reason": "Transformers package or pretrained model unavailable."}

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT epoch 1/2: loss=0.2162; val macro F1=0.7420
BERT epoch 2/2: loss=0.1550; val macro F1=0.7836


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 9. Results and export

In [10]:
summary_rows = []
for model_name, model_result in results.items():
    if model_result.get("status") == "not_run":
        summary_rows.append({"model": model_name, "threshold": "—", "micro F1": "not run", "macro F1": "—"})
        continue
    for threshold_name, key in [("fixed 0.5", "fixed_0_5"), ("validation-tuned", "validation_tuned")]:
        metric = model_result[key]
        summary_rows.append({"model": model_name, "threshold": threshold_name,
                             "micro F1": metric["micro_f1"], "macro F1": metric["macro_f1"],
                             "exact match": metric["exact_match_accuracy"],
                             "Hamming loss": metric["hamming_loss"]})
display(pd.DataFrame(summary_rows))

# Save all measured metrics for reproducibility.
(OUT_DIR / "notebook_metrics.json").write_text(json.dumps({
    "seed": SEED, "split_sizes": {"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)},
    "labels": LABELS, "models": results,
}, indent=2))

,model,threshold,micro F1,macro F1,exact match,Hamming loss
0,TF-IDF + logistic regression,fixed 0.5,0.815511,0.578087,0.674452,0.073324
1,TF-IDF + logistic regression,validation-tuned,0.831888,0.750172,0.683508,0.070861
2,BiLSTM,fixed 0.5,0.775835,0.521871,0.622974,0.091675
3,BiLSTM,validation-tuned,0.783283,0.559917,0.621544,0.091039
4,BiGRU,fixed 0.5,0.745524,0.493493,0.586749,0.100493
5,BiGRU,validation-tuned,0.744226,0.505735,0.493327,0.118764
6,BERT,fixed 0.5,0.811923,0.741213,0.669209,0.077693
7,BERT,validation-tuned,0.816583,0.742243,0.653480,0.078726


17187

## References

- Hochreiter, S., & Schmidhuber, J. (1997). Long Short-Term Memory. *Neural Computation, 9*(8), 1735–1780. https://doi.org/10.1162/neco.1997.9.8.1735
- Cho, K., et al. (2014). Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation. *EMNLP 2014*, 1724–1734. https://doi.org/10.3115/v1/D14-1179
- Devlin, J., et al. (2019). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. *NAACL-HLT 2019*, 4171–4186. https://doi.org/10.18653/v1/N19-1423
- FastText vectors: official 300-D English wiki-news subword vectors and citation information: https://fasttext.cc/docs/en/english-vectors.html
- Dataset: *Topic Modeling for Research Articles* (Kaggle); matching record: https://zenodo.org/records/7671877
